# ExioML Factor Loading and Decision Tree Example

This notebook demonstrates how to load PxP emission factor data via the PyPI-distributed `exioml` package and train a `scikit-learn` decision tree regressor whose hyperparameters are tuned with a parameter grid.

In [1]:
import importlib
import subprocess
import sys

package_name = "exioml"
try:
    importlib.import_module(package_name)
    print("exioml is already installed and ready to use.")
except ImportError:
    print("exioml is missing; installing via pip...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


exioml is already installed and ready to use.


## Import APIs and regression dependencies

In [2]:
from exioml import load_factor, list_regions, list_years
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

pd.options.display.precision = 3


## Query available PxP regions and years

In [3]:
years = list_years("PxP")
regions = list_regions("PxP")
print(f"PxP mode includes {len(years)} years: {years}")
print(f"PxP mode includes {len(regions)} regions: {regions}")


PxP mode includes 28 years: [1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
PxP mode includes 49 regions: ['AT', 'AU', 'BE', 'BG', 'BR', 'CA', 'CH', 'CN', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'ID', 'IE', 'IN', 'IT', 'JP', 'KR', 'LT', 'LU', 'LV', 'MT', 'MX', 'NL', 'NO', 'PL', 'PT', 'RO', 'RU', 'SE', 'SI', 'SK', 'TR', 'TW', 'US', 'WA', 'WE', 'WF', 'WL', 'WM', 'ZA']


## Load a sample dataset

The snippet below selects four regions and three years via `exioml.load_factor`, and fetches additional value-added, employment, and energy columns for regression.

In [4]:
YEARS = [2010, 2011, 2012]
REGIONS = ["US", "CN", "DE", "JP"]
EXTRA_COLUMNS = ["value_added_meur", "employment_k", "energy_carrier_tj"]

frame = load_factor(
    schema="PxP",
    years=YEARS,
    regions=REGIONS,
    columns=EXTRA_COLUMNS,
)
frame = frame.dropna(subset=EXTRA_COLUMNS + ["factor_value"])
numeric_columns = frame.select_dtypes(include=[np.number]).columns.tolist()
frame[numeric_columns] = frame[numeric_columns].apply(lambda col: np.log1p(col))
print(f"Log-transformed columns: {numeric_columns}")
print(f"Samples loaded: {len(frame)}")
frame.head()


Log-transformed columns: ['year', 'ghg_emissions', 'factor_value', 'value_added_meur', 'employment_k', 'energy_carrier_tj']
Samples loaded: 1975


,schema,region,sector,year,ghg_emissions,factor_value,value_added_meur,employment_k,energy_carrier_tj
0,PxP,DE,Wheat,7.606,22.634,22.634,7.904,3.924,3.555
1,PxP,DE,Cereal grains nec,7.606,22.466,22.466,7.701,3.796,8.433
2,PxP,DE,"Vegetables, fruit, nuts",7.606,20.571,20.571,8.569,5.132,5.361
3,PxP,DE,Oil seeds,7.606,21.890,21.890,7.197,4.342,2.214
4,PxP,DE,"Sugar cane, sugar beet",7.606,20.633,20.633,6.115,2.350,5.775


## Split the training and test sets

In [5]:
categorical_features = ["region", "sector"]
numeric_features = ["year", "employment_k", "energy_carrier_tj", "value_added_meur"]
feature_columns = categorical_features + numeric_features

X = frame[feature_columns]
y = frame["factor_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train shape: {X_train.shape}, test shape: {X_test.shape}")
X_train.head()


Train shape: (1580, 6), test shape: (395, 6)


,region,sector,year,employment_k,energy_carrier_tj,value_added_meur
198,US,Crude petroleum and services related to crude ...,7.606,3.541,13.367,10.658
990,US,Textiles waste for treatment: incineration,7.607,1.798,8.334,6.280
1644,US,Other business services (74),7.607,9.057,13.828,14.018
332,US,Wood waste for treatment: incineration,7.606,1.873,8.235,6.337
1340,DE,Lignite/Brown Coal,7.607,1.901,9.767,6.510


## Define the decision tree + parameter grid search

`GridSearchCV` feeds a parameter grid into the pipeline so that the optimal tree depth, minimum split size, and minimum leaf size are selected automatically.

In [6]:
preprocess = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", DecisionTreeRegressor(random_state=42)),
    ]
)

param_grid = {
    "model__max_depth": [4, 6, None],
    "model__min_samples_split": [2, 10],
    "model__min_samples_leaf": [1, 5],
}

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=-1,
    verbose=0,
)

search.fit(X_train, y_train)
print("Best parameters:", search.best_params_)
print("Cross-validation MSE:", -search.best_score_)


Best parameters: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2}
Cross-validation MSE: 1.356110334597222


## Evaluate on the held-out test set

In [7]:
best_model = search.best_estimator_
predictions = best_model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
print(f"Test-set MSE: {mse:.4e}")

results = (
    pd.DataFrame(search.cv_results_)
    .loc[:, ["params", "mean_test_score", "rank_test_score"]]
    .sort_values("rank_test_score")
)
results.head()


Test-set MSE: 4.8324e-01


,params,mean_test_score,rank_test_score
8,"{'model__max_depth': None, 'model__min_samples...",-1.356,1
9,"{'model__max_depth': None, 'model__min_samples...",-1.835,2
5,"{'model__max_depth': 6, 'model__min_samples_le...",-2.302,3
4,"{'model__max_depth': 6, 'model__min_samples_le...",-2.474,4
0,"{'model__max_depth': 4, 'model__min_samples_le...",-2.587,5
